In [1]:
from TransformerLensResidualContributors import *

tlmodel = TransformerLensResidualContributors("gpt2")

/Users/isaiah/repos/probing-utils/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  mps


In [2]:
toks = "These are some tokens that represent an input"

out = tlmodel.Run(toks)
print(out.shape)

torch.Size([1, 9, 25, 768])


In [3]:
import datasets 
test_ds = dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech', 'default', columns=['text','sentiment','respect','insult','humiliate','status','dehumanize','violence','genocide', 'attack_defend','hatespeech'])
test_ds = test_ds.with_format('torch')
print(test_ds['train'].features['sentiment'])

Value('float64')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [4]:
activations = tlmodel.activations
print(activations)

['hook_embed', 'blocks.0.hook_attn_out', 'blocks.0.hook_mlp_out', 'blocks.1.hook_attn_out', 'blocks.1.hook_mlp_out', 'blocks.2.hook_attn_out', 'blocks.2.hook_mlp_out', 'blocks.3.hook_attn_out', 'blocks.3.hook_mlp_out', 'blocks.4.hook_attn_out', 'blocks.4.hook_mlp_out', 'blocks.5.hook_attn_out', 'blocks.5.hook_mlp_out', 'blocks.6.hook_attn_out', 'blocks.6.hook_mlp_out', 'blocks.7.hook_attn_out', 'blocks.7.hook_mlp_out', 'blocks.8.hook_attn_out', 'blocks.8.hook_mlp_out', 'blocks.9.hook_attn_out', 'blocks.9.hook_mlp_out', 'blocks.10.hook_attn_out', 'blocks.10.hook_mlp_out', 'blocks.11.hook_attn_out', 'blocks.11.hook_mlp_out']


In [9]:
import torch
import numpy as np

# Define the order of your numeric columns
NUMERIC_COLS = ['sentiment','respect','insult','humiliate','status','dehumanize','violence','genocide', 'attack_defend','hatespeech']

# Use map to combine numeric columns into one
def combine_numeric_features(example):
    # Stack numeric columns in the specified order
    features = [example[col] for col in NUMERIC_COLS]
    features = torch.stack(features)
    example['numeric_features'] = features.numpy()
    return example

#test_ds = test_ds.map(combine_numeric_features)

# Set format with explicit column ordering
#test_ds.set_format(
#    type='torch',
#    columns=['text', 'numeric_features']  # Columns in your desired order
#)

def collate_fn(batch):
    numeric_features = torch.stack([
        torch.tensor([item[col] for col in NUMERIC_COLS], dtype=torch.float32) for item in batch])
    return {'text':[item['text'] for item in batch], 'numeric_features': numeric_features}

dataloader = torch.utils.data.DataLoader(test_ds['train'], collate_fn=collate_fn)

print(dataloader)


In [6]:
text = test_ds['train'][0]['text']
print(text)
print(test_ds['train'][0])
print(test_ds['train'])

Yes indeed. She sort of reminds me of the elder lady that played the part in the movie "Titanic" who was telling her story!!! And I wouldn't have wanted to cover who I really am!! I would be proud!!!! WE should be proud of our race no matter what it is!!
{'sentiment': tensor(0.), 'respect': tensor(0.), 'insult': tensor(0.), 'humiliate': tensor(0.), 'status': tensor(2.), 'dehumanize': tensor(0.), 'violence': tensor(0.), 'genocide': tensor(0.), 'attack_defend': tensor(0.), 'hatespeech': tensor(0.), 'text': 'Yes indeed. She sort of reminds me of the elder lady that played the part in the movie "Titanic" who was telling her story!!! And I wouldn\'t have wanted to cover who I really am!! I would be proud!!!! WE should be proud of our race no matter what it is!!'}
Dataset({
    features: ['sentiment', 'respect', 'insult', 'humiliate', 'status', 'dehumanize', 'violence', 'genocide', 'attack_defend', 'hatespeech', 'text'],
    num_rows: 135556
})


In [7]:
out = tlmodel.Run(text)
print(out.shape)
print(tlmodel.ActivationShape())
print(tlmodel.tl_model.to_tokens(text).shape)

torch.Size([1, 62, 25, 768])
torch.Size([25, 768])
torch.Size([1, 62])


In [11]:
from tqdm import tqdm
for i, x in tqdm(enumerate(dataloader)):
    print(x)
    if(i > 5):
        break

6it [00:00, 619.15it/s]

{'text': ['Yes indeed. She sort of reminds me of the elder lady that played the part in the movie "Titanic" who was telling her story!!! And I wouldn\'t have wanted to cover who I really am!! I would be proud!!!! WE should be proud of our race no matter what it is!!'], 'numeric_features': tensor([[0., 0., 0., 0., 2., 0., 0., 0., 0., 0.]])}
{'text': ['The trans women reading this tweet right now is beautiful'], 'numeric_features': tensor([[0., 0., 0., 0., 2., 0., 0., 0., 2., 0.]])}
{'text': ["Question: These 4 broads who criticize America, what country did they flee to get here? And now they want to make OUR America like THEIR former HELL HOLE. I don't think so!!!!!!!!!!  Let them explain their GRATITUDE for letting them in OUR country."], 'numeric_features': tensor([[4., 4., 4., 4., 4., 4., 0., 0., 4., 2.]])}
{'text': ['It is about time for all illegals to go back to their country of origin and keep our freeway open and prevent heavy traffic.'], 'numeric_features': tensor([[2., 3., 2.,

In [46]:
class StackedLinear(torch.nn.Module):
    def __init__(self, layers:int, in_dim:int, out_dim:int):
        super(StackedLinear,self).__init__()
        self.weights = torch.nn.Parameter(torch.randn(layers,in_dim,out_dim))

    def forward(self, x):
        return torch.einsum('bplx,lxy->bply',x,self.weights)

In [61]:
#device = tlmodel.tl_model.device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
probe = StackedLinear(25,768,10).to(device)
#print(probe.parameters())
for param in probe.parameters():
    print(param.device)
sgd = torch.optim.SGD(probe.parameters(), lr = 1e-4)
pbar = tqdm(enumerate(dataloader), total=len(dataloader))
for i, x in pbar:
    text, label = x['text'], x['numeric_features'].to(device)
    sgd.zero_grad()
    activations = tlmodel.Run(text).to(device)
    probe_output = probe(activations)
    label = label.unsqueeze(0).unsqueeze(0).expand(probe_output.shape)    
    loss = torch.nn.MSELoss()(probe_output, label)

    loss.backward()
    sgd.step()
    pbar.set_description(f'Step {i} Loss: {loss.item():.4f}')
    if( i > 500):
        break

mps:0


Step 501 Loss: 603.3124:   0%|▏                                                    | 501/135556 [00:08<36:22, 61.89it/s]


In [1]:
from SpecificLabeledHFDataset import *
from tqdm import tqdm
class StackedLinear(torch.nn.Module):
    def __init__(self, layers:int, in_dim:int, out_dim:int):
        super(StackedLinear,self).__init__()
        self.weights = torch.nn.Parameter(torch.randn(layers,in_dim,out_dim))

    def forward(self, x):
        return torch.einsum('bplx,lxy->bply',x,self.weights)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
probe = StackedLinear(25,768,10).to(device)
for param in probe.parameters():
    print(param.device)
sgd = torch.optim.SGD(probe.parameters(), lr = 1e-4)

ds = SpecificLabeledHFDataset()

pbar = tqdm(enumerate(ds), total=len(ds))
for i, (activations, labels) in pbar:
    sgd.zero_grad()
    probe_output = probe(activations)
    loss = torch.nn.MSELoss()(probe_output, labels)

    loss.backward()
    sgd.step()
    pbar.set_description(f'Step {i} Loss: {loss.item():.4f}')
    if(i > 500):
        print("breaking")
        break
print("Here")

/Users/isaiah/repos/probing-utils/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mps:0


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  mps


Step 501 Loss: 625.4171:   0%|▏                                                    | 501/135556 [00:09<42:22, 53.12it/s]

breaking
Here


In [1]:
from TrainingRunBuilder import *
from SpecificLabeledHFDataset import *
import torch
dummy_param = torch.nn.Parameter(torch.randn(1))
ds = SpecificLabeledHFDataset()
run = (TrainingRunBuilder()
    .use_dataset(ds)
    .use_probe("linear")
    .use_optimizer(torch.optim.SGD([dummy_param],lr=1e-3))
    .use_loss("mse")
      ).build()
run.Run()

/Users/isaiah/repos/probing-utils/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  mps
mps


 50%|█████████████████████████████████████▊                                      | 67382/135556 [18:26<18:39, 60.88it/s]


KeyboardInterrupt: 

In [2]:
run.Run()

AttributeError: 'TrainingRun' object has no attribute 'initialize'